In [63]:
import numpy as np
import cv2
import pickle
import struct
import bz2
import os
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn

with open('phase3_results.pkl', 'rb') as f:
    p3 = pickle.load(f)

img = p3['img']
height, width = p3['height'], p3['width']
region_map = p3['region_map']
special_regions = p3['special_regions']
gradient_regions = p3['gradient_regions']
flat_regions = p3['flat_regions']

output_dir = "comic/phase4_output"
os.makedirs(output_dir, exist_ok=True)
num_pixels = height * width

# Keep only region IDs that actually exist in region_map,
# and recover any uncategorized region into flat_regions.
actual_regions = set(int(r) for r in np.unique(region_map[region_map >= 0]))
special_regions = set(int(r) for r in special_regions) & actual_regions
gradient_regions = {int(k): v for k, v in gradient_regions.items() if int(k) in actual_regions}
flat_set = set(int(r) for r in flat_regions) & actual_regions

categorized = special_regions | set(gradient_regions.keys()) | flat_set
uncategorized = actual_regions - categorized
flat_set |= uncategorized
flat_regions = list(flat_set)

assert (special_regions | set(gradient_regions.keys()) | flat_set) == actual_regions

print(f"Special: {len(special_regions)}  Gradient: {len(gradient_regions)}  Flat: {len(flat_regions)}")
print(f"Recovered uncategorized regions: {len(uncategorized)}")

Special: 10848  Gradient: 308  Flat: 2502
Recovered uncategorized regions: 97


In [64]:
# Palette only covers gradient and flat regions now.
# Special regions store exact per-pixel colors instead of a palette index.
palette_regions = set(gradient_regions.keys()) | set(flat_regions)

region_avg_colors = {}
for rid in palette_regions:
    mask = (region_map == rid)
    region_avg_colors[rid] = np.mean(img[mask], axis=0)

ptol = 3  # Palette color tolerance
palette = []
region_to_palette = {}

sorted_rids = sorted(palette_regions, key=lambda r: np.sum(region_map == r), reverse=True)

for rid in sorted_rids:
    color = region_avg_colors[rid]
    best_idx = -1
    for idx, (pal_color, pal_count) in enumerate(palette):
        if np.sqrt(np.sum((pal_color - color) ** 2)) < ptol:
            best_idx = idx
            break
    if best_idx >= 0:
        pal_color, pal_count = palette[best_idx]
        palette[best_idx] = ((pal_color * pal_count + color) / (pal_count + 1), pal_count + 1)
        region_to_palette[rid] = best_idx
    else:
        palette.append((color, 1))
        region_to_palette[rid] = len(palette) - 1

print(f"Palette generated: {len(palette)} colors (for gradient + flat regions)")

Palette generated: 982 colors (for gradient + flat regions)


In [65]:
def build_all_rle_runs(region_map, height, width):
    """
    Single pass over region_map that builds a run-length list for every
    region. Each run is (row, start_col, length), meaning pixels
    region_map[row, start_col : start_col+length] all belong to that region.

    RLE stores pixel membership directly, which is exact for any region
    shape, including thin strokes (text/outline regions), unlike boundary
    chain-code tracing which fails on 1-pixel-wide shapes.
    """
    runs_by_region = {}
    for y in range(height):
        row = region_map[y]
        x = 0
        while x < width:
            rid = int(row[x])
            if rid < 0:
                x += 1
                continue
            start = x
            while x < width and row[x] == rid:
                x += 1
            runs_by_region.setdefault(rid, []).append((y, start, x - start))
    return runs_by_region

all_regions = special_regions | set(gradient_regions.keys()) | set(flat_regions)
all_runs = build_all_rle_runs(region_map, height, width)

total_runs = sum(len(all_runs.get(rid, [])) for rid in all_regions)
print(f"RLE runs built for {len(all_regions)} regions")
print(f"Total runs: {total_runs}")

RLE runs built for 13658 regions
Total runs: 49656


In [66]:
data = bytearray()

# Header: dimensions + palette size
data.extend(struct.pack('>HH', height, width))
data.extend(struct.pack('>H', len(palette)))
for color, count in palette:
    r, g, b = np.clip(color, 0, 255).astype(np.uint8)
    data.extend(struct.pack('BBB', int(r), int(g), int(b)))

total_regions = len(special_regions) + len(gradient_regions) + len(flat_regions)
data.extend(struct.pack('>I', total_regions))

def write_runs_header(data, runs):
    """Write run count and (row, start_col, length) for each run, without color data."""
    data.extend(struct.pack('>I', len(runs)))
    for row, start_col, length in runs:
        data.extend(struct.pack('>HHH', row, start_col, length))

# Special regions - type 2
# MRC-inspired: exact pixel shape (RLE) + exact per-pixel original color.
# This keeps text/outline regions fully lossless, avoiding the ringing
# artifacts that lossy methods like JPEG produce on sharp text edges.
for rid in special_regions:
    runs = all_runs.get(rid, [])
    data.extend(struct.pack('B', 2))
    write_runs_header(data, runs)
    # Store the true RGB color for every pixel in every run
    for row, start_col, length in runs:
        for x in range(start_col, start_col + length):
            r, g, b = img[row, x]
            data.extend(struct.pack('BBB', int(r), int(g), int(b)))

# Gradient regions - type 1 (direction + start/end colors + exact pixel runs)
for rid, grad_info in gradient_regions.items():
    runs = all_runs.get(rid, [])
    direction = grad_info['direction']
    start_color = np.clip(grad_info['start_color'], 0, 255).astype(np.uint8)
    end_color = np.clip(grad_info['end_color'], 0, 255).astype(np.uint8)
    data.extend(struct.pack('B', 1))
    data.extend(struct.pack('B', int(direction)))
    data.extend(struct.pack('BBB', *start_color.tolist()))
    data.extend(struct.pack('BBB', *end_color.tolist()))
    write_runs_header(data, runs)

# Flat regions - type 0 (palette color + exact pixel runs)
for rid in flat_regions:
    runs = all_runs.get(rid, [])
    pal_idx = region_to_palette.get(rid, 0)
    data.extend(struct.pack('B', 0))
    data.extend(struct.pack('>H', pal_idx))
    write_runs_header(data, runs)

chain_data = bytes(data)
compressed = bz2.compress(chain_data, 9)
size = len(compressed)
bpp = size * 8 / num_pixels

print(f"Encoded data size: {len(chain_data)} bytes")
print(f"Compressed size:   {size} bytes")
print(f"Compression ratio: {len(chain_data) / size:.2f}x")
print(f"BPP:               {bpp:.4f}")

Encoded data size: 546580 bytes
Compressed size:   309358 bytes
Compression ratio: 1.77x
BPP:               7.8717


In [67]:
phase4_results = {
    'img': img,               # kept only for comparison in Phase 5, NOT used by decoder
    'height': height,
    'width': width,
    'compressed_data': compressed,
    'size': size,
    'bpp': bpp
}

with open('phase4_results.pkl', 'wb') as f:
    pickle.dump(phase4_results, f)

print("Phase 4 results saved to phase4_results.pkl")
print(f"Compressed size: {size} bytes | BPP: {bpp:.4f}")

Phase 4 results saved to phase4_results.pkl
Compressed size: 309358 bytes | BPP: 7.8717
